# Parcellated ISPC — Left-Wing Subjects: Agree vs. Disagree

**Goal**: Test whether neural alignment (ISPC) is higher when left-wing
subjects view content they politically *agree* with versus content they
*disagree* with.

**Condition grouping**:
- **AGREE** = `ProLeft` + `AntiRight` (content that matches left-wing views)
- **DISAGREE** = `AntiLeft` + `ProRight` (content that contradicts left-wing views)

**Contrast**: AGREE > DISAGREE, tested with a split-pool permutation null
distribution (1 000 iterations) and Benjamini–Hochberg FDR correction across
parcels (Schaefer 2018 400-parcel + Tian S3 subcortex).

In [ ]:
from pathlib import Path

# Existing parcellated-ISC API (do not modify this file)
from yy_fmri_kit.event_isc.extraction.parcel import (
    Config,
    load_events,
    load_timeseries,
    extract_post_patterns,
    compute_isc,
    fdr_correct,
    results_to_dataframe,
)

# New contrast utilities
from yy_fmri_kit.event_isc.contrast import (
    filter_subjects_by_group,
    merge_conditions,
    contrast_permutation_test,
)

## 1. Subject Filtering & Configuration

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
ROOT            = Path("/path/to/project")
BEHAVIORAL_CSV  = ROOT / "behavioral_analyses/data/250226/merged_behavioral_bids.csv"
EVENTS_CSV      = ROOT / "behavioral_analyses/data/130426/summary_with_bids_ids.csv"
DATA_DIR        = ROOT / "data/derivatives/parcellated_tian"
OUTPUT_DIR      = ROOT / "data/derivatives/ispc_leftwing_contrast"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── subject selection ──────────────────────────────────────────────────────
# Reads merged_behavioral_bids.csv and keeps only political_group == "left"
left_subs = filter_subjects_by_group(
    BEHAVIORAL_CSV,
    group="left",
    id_col="bids_id",
    group_col="political_group",
)

# ── analysis config ────────────────────────────────────────────────────────
# tsv_glob must match the actual filename pattern under DATA_DIR:
#   sub-N/sub-N_ses-*_task-{run_type}_*atlas-Schaefer2018*timeseries.tsv
cfg = Config(
    data_dir     = DATA_DIR,
    events_csv   = EVENTS_CSV,
    subjects     = left_subs,
    run_types    = ["AntiLeft", "AntiRight", "ProLeft", "ProRight"],
    tr           = 1.0,
    shift_tr     = 4,
    tsv_glob     = "{subject}/{subject}_*_task-{run_type}_*atlas-Schaefer2018*timeseries.tsv",
    subject_col  = "bids_id",
    run_col      = "run",
    post_col     = "post_id",
    onset_col    = "onset_s",
    duration_col = "duration_s",
    n_perms      = 1000,
    fdr_q        = 0.05,
    seed         = 42,
)

print(f"Subjects in analysis: {len(cfg.subjects)}")
print(f"Conditions: {cfg.run_types}")

## 2. Load Events & Timeseries, Extract Post Patterns

In [ ]:
events_df = load_events(cfg)
print(events_df.head())

In [ ]:
# ts_dict: {(subject, run_type): DataFrame(n_trs, n_parcels)}
ts_dict = load_timeseries(cfg)

# parcel names come from the TSV column headers
first_key = next(iter(ts_dict))
parcel_names = ts_dict[first_key].columns.tolist()
print(f"Parcels: {len(parcel_names)}")
print(f"Example parcels: {parcel_names[:5]} ... {parcel_names[-5:]}")

In [ ]:
# patterns: {run_type: {post_id: array(n_subjects, n_parcels)}}
patterns = extract_post_patterns(ts_dict, events_df, cfg)

for rt, posts in patterns.items():
    n_subj = next(iter(posts.values())).shape[0]
    print(f"  {rt}: {len(posts)} posts, {n_subj} subjects")

## 3. Merge Conditions into AGREE / DISAGREE

In [ ]:
CONDITION_MAP = {
    "agree":    ["ProLeft",  "AntiRight"],   # content left-wing subjects agree with
    "disagree": ["AntiLeft", "ProRight"],    # content they disagree with
}

merged = merge_conditions(patterns, CONDITION_MAP)

# sanity: total post count should equal sum across all four conditions
total_merged = sum(len(v) for v in merged.values())
total_orig   = sum(len(v) for v in patterns.values())
assert total_merged == total_orig, (
    f"Post count mismatch after merging: {total_merged} vs {total_orig}"
)
print(f"\nTotal posts (merged): {total_merged} == original {total_orig} ✓")

## 4. Compute ISPC per Condition Group

In [ ]:
isc_agree,    isc_agree_subj    = compute_isc(merged["agree"])
isc_disagree, isc_disagree_subj = compute_isc(merged["disagree"])

print(f"ISPC AGREE    — mean: {isc_agree.mean():.4f},  max: {isc_agree.max():.4f}")
print(f"ISPC DISAGREE — mean: {isc_disagree.mean():.4f},  max: {isc_disagree.max():.4f}")
print(f"\nObserved contrast (agree - disagree): {(isc_agree - isc_disagree).mean():.4f}")

## 5. Contrast Permutation Test (AGREE > DISAGREE)

In [ ]:
# Split-pool null: pool all posts, randomly re-assign to agree/disagree groups,
# compute ISC difference — repeated 1 000 times.
obs_contrast, p_vals, null_dist = contrast_permutation_test(
    merged["agree"],
    merged["disagree"],
    n_perms=cfg.n_perms,
    seed=cfg.seed,
)

print(f"null_dist shape: {null_dist.shape}")

## 6. FDR Correction

In [ ]:
rejected, p_fdr = fdr_correct(p_vals, q=cfg.fdr_q)

print(f"Significant parcels (FDR q={cfg.fdr_q}): {rejected.sum()} / {len(rejected)}")

if rejected.any():
    sig_idx = rejected.nonzero()[0]
    for i in sig_idx:
        print(f"  {parcel_names[i]:50s}  r={obs_contrast[i]:.4f}  p_fdr={p_fdr[i]:.4f}")

## 7. Save Results

In [ ]:
import numpy as np

# Tidy results DataFrame.
# We do NOT pass subj_vals/subjects: isc_agree_subj rows correspond only to
# subjects with valid data in BOTH ProLeft and AntiRight, which may be fewer
# than len(cfg.subjects). Passing cfg.subjects would cause an IndexError.
# Per-group ISC is attached as separate columns below instead.
df = results_to_dataframe(
    parcel_names,
    obs_contrast,
    p_vals,
    rejected,
    p_fdr,
)

# Rename the generic 'r' column to clarify it is the contrast statistic
df = df.rename(columns={"r": "obs_contrast"})

# Attach raw per-group ISC so downstream analyses have the full picture
df["isc_agree"]    = isc_agree
df["isc_disagree"] = isc_disagree

out_csv = OUTPUT_DIR / "agree_vs_disagree_contrast.csv"
df.to_csv(out_csv, index=False)
print(f"Results saved → {out_csv}")

# Save null distribution for later visualisation or additional tests
np.save(OUTPUT_DIR / "null_dist_agree_vs_disagree.npy", null_dist)
print(f"Null distribution saved → {OUTPUT_DIR / 'null_dist_agree_vs_disagree.npy'}")

df.head()

## 8. Summary Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: mean ISC per condition group ─────────────────────────────────────
ax = axes[0]
means = [isc_agree.mean(), isc_disagree.mean()]
sems  = [isc_agree.std() / np.sqrt(len(isc_agree)),
         isc_disagree.std() / np.sqrt(len(isc_disagree))]
bars = ax.bar(["Agree", "Disagree"], means, yerr=sems,
              color=["#2196F3", "#F44336"], capsize=5)
ax.set_ylabel("Mean ISPC (r)")
ax.set_title("ISPC by Condition Group\n(mean ± SEM across parcels)")
ax.axhline(0, color="k", lw=0.8, ls="--")

# ── Right: contrast distribution & top parcels ─────────────────────────────
ax = axes[1]
ax.hist(obs_contrast, bins=40, color="#9C27B0", alpha=0.7, label="All parcels")
if rejected.any():
    ax.hist(obs_contrast[rejected], bins=20, color="#FF9800",
            alpha=0.9, label=f"Significant (n={rejected.sum()})")
ax.axvline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("Observed contrast (agree − disagree ISPC)")
ax.set_ylabel("Parcel count")
ax.set_title("Contrast Distribution\n(FDR-significant parcels highlighted)")
ax.legend()

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "agree_vs_disagree_summary.png", dpi=150)
plt.show()
print(f"Figure saved → {OUTPUT_DIR / 'agree_vs_disagree_summary.png'}")

In [ ]:
# Null distribution for the parcel with the largest observed contrast
top_idx = int(obs_contrast.argmax())
top_parcel = parcel_names[top_idx]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_dist[:, top_idx], bins=40, color="grey", alpha=0.7, label="Null")
ax.axvline(obs_contrast[top_idx], color="red", lw=2,
           label=f"Observed = {obs_contrast[top_idx]:.4f}")
ax.set_xlabel("Contrast (agree − disagree ISPC)")
ax.set_ylabel("Permutation count")
ax.set_title(f"Null distribution — top parcel\n{top_parcel}\np_fdr = {p_fdr[top_idx]:.4f}")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Brain Surface Maps (yabplot)

Visualise ISPC values and the agree–disagree contrast on the cortical surface
using the `schaefer400` atlas bundled with **yabplot**.

> **Note**: 342 of 400 Schaefer parcel names match exactly between our atlas
> version and yabplot’s internal atlas. The remaining 58 parcels appear in the
> background (`nan_color`) due to minor cross-version naming differences.
> Subcortical Tian-S3 parcels are not visualised here (yabplot has no Tian atlas).

In [ ]:
import yabplot as yab
import numpy as np

# yabplot covers the first 400 Schaefer cortical parcels only.
# The last 32 columns are Tian-S3 subcortical — no matching atlas in yabplot.
cortical_parcels = parcel_names[:400]

# Build region → value dicts; dict API silently skips unrecognised names
isc_agree_dict    = {cortical_parcels[i]: float(isc_agree[i])    for i in range(400)}
isc_disagree_dict = {cortical_parcels[i]: float(isc_disagree[i]) for i in range(400)}

# Shared colour range so both maps are directly comparable
vmin = min(float(np.nanmin(isc_agree[:400])),    float(np.nanmin(isc_disagree[:400])))
vmax = max(float(np.nanmax(isc_agree[:400])),    float(np.nanmax(isc_disagree[:400])))

views = ['left_lateral', 'left_medial', 'right_lateral', 'right_medial']

print('=== AGREE ISPC ===')
yab.plot_cortical(
    data=isc_agree_dict,
    atlas='schaefer400',
    views=views,
    cmap='Reds',
    vminmax=[vmin, vmax],
    figsize=(1200, 400),
    display_type='static',
    export_path=str(OUTPUT_DIR / 'brain_isc_agree.png'),
)

print('\n=== DISAGREE ISPC ===')
yab.plot_cortical(
    data=isc_disagree_dict,
    atlas='schaefer400',
    views=views,
    cmap='Reds',
    vminmax=[vmin, vmax],
    figsize=(1200, 400),
    display_type='static',
    export_path=str(OUTPUT_DIR / 'brain_isc_disagree.png'),
)

In [ ]:
# Contrast: Agree − Disagree
contrast_dict = {cortical_parcels[i]: float(obs_contrast[i]) for i in range(400)}

# Symmetric range centred at zero for coolwarm
cmax = float(np.nanmax(np.abs(obs_contrast[:400])))

print('=== CONTRAST (agree − disagree) — all parcels ===')
yab.plot_cortical(
    data=contrast_dict,
    atlas='schaefer400',
    views=views,
    cmap='coolwarm',
    vminmax=[-cmax, cmax],
    figsize=(1200, 400),
    display_type='static',
    export_path=str(OUTPUT_DIR / 'brain_contrast.png'),
)

# Non-significant parcels masked as NaN → rendered in light grey
contrast_sig_dict = {
    cortical_parcels[i]: float(obs_contrast[i]) if rejected[i] else float('nan')
    for i in range(400)
}

print('\n=== CONTRAST — FDR-significant parcels only ===')
yab.plot_cortical(
    data=contrast_sig_dict,
    atlas='schaefer400',
    views=views,
    cmap='coolwarm',
    vminmax=[-cmax, cmax],
    nan_color=(0.92, 0.92, 0.92),
    figsize=(1200, 400),
    display_type='static',
    export_path=str(OUTPUT_DIR / 'brain_contrast_significant.png'),
)

## 10. Brain Map Helper (local atlas → surface projection)

All brain visualisations below use the local Schaefer+Tian atlas NIfTI to
map parcel values into a 3-D volume and project it onto the fsLR-32k
midthickness surface via `yabplot.project_vol2surf`. This completely avoids
the name-matching issue and covers **all 432 parcels** (400 Schaefer + 32 Tian S3).

In [ ]:
import tempfile
import yabplot as yab
import yabplot.data as ydata
import numpy as np
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti

ATLAS_NII  = ROOT / 'data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz'
LABELS_TSV = ROOT / 'data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv'

_lh_surf, _rh_surf = ydata.get_surface_paths('midthickness', 'bmesh')

def brain_map(
    values, label, *,
    cmap='coolwarm', vminmax=(None, None),
    nan_color=(0.92, 0.92, 0.92),
):
    """
    Map parcel values → NIfTI → surface projection → yabplot figure.
    Saves PNG to OUTPUT_DIR/{label}.png.
    """
    tmp = Path(tempfile.mktemp(suffix='.nii.gz'))
    parcels_to_nifti(values, parcel_names, ATLAS_NII, LABELS_TSV, tmp)
    lh_data, rh_data = yab.project_vol2surf(str(tmp), interpolation='nearest')
    tmp.unlink(missing_ok=True)
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    views = ['left_lateral', 'left_medial', 'right_lateral', 'right_medial']
    return yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views=views,
        cmap=cmap,
        vminmax=list(vminmax),
        nan_color=nan_color,
        figsize=(1200, 400),
        display_type='static',
        export_path=str(OUTPUT_DIR / f'{label}.png'),
    )

print('brain_map helper ready.')

## 11. Within-AGREE Contrast: AntiRight vs ProLeft

Both conditions contain content that left-wing subjects *agree* with, but
they differ in framing:
- **AntiRight** — outgroup-derogating (criticising the right)
- **ProLeft** — ingroup-affirming (supporting the left)

The contrast tests whether outgroup derogation (`AntiRight`) elicits
stronger neural alignment than ingroup affirmation (`ProLeft`).

**Method**: Paired t-test across the 23 left-wing subjects' per-subject ISC
values, one t-test per parcel; Benjamini–Hochberg FDR correction across parcels.

In [ ]:
from yy_fmri_kit.event_isc.contrast import ttest_contrast
from yy_fmri_kit.event_isc.extraction.parcel import fdr_correct, results_to_dataframe

t_ar_pl, p_ar_pl, mu_ar, mu_pl = ttest_contrast(
    patterns['AntiRight'], patterns['ProLeft']
)
rejected_ar_pl, p_fdr_ar_pl = fdr_correct(p_ar_pl, q=cfg.fdr_q)

print(f'Significant parcels (FDR q={cfg.fdr_q}): {rejected_ar_pl.sum()} / {len(rejected_ar_pl)}')
if rejected_ar_pl.any():
    for i in rejected_ar_pl.nonzero()[0]:
        print(f'  {parcel_names[i]:50s}  t={t_ar_pl[i]:.3f}  p_fdr={p_fdr_ar_pl[i]:.4f}')

In [ ]:
# Save results
df_ar_pl = results_to_dataframe(parcel_names, t_ar_pl, p_ar_pl, rejected_ar_pl, p_fdr_ar_pl)
df_ar_pl = df_ar_pl.rename(columns={'r': 't_stat'})
df_ar_pl['isc_anti_right'] = mu_ar
df_ar_pl['isc_pro_left']   = mu_pl
df_ar_pl['isc_diff']       = mu_ar - mu_pl
df_ar_pl.to_csv(OUTPUT_DIR / 'antiright_vs_proleft_ttest.csv', index=False)
print('Saved antiright_vs_proleft_ttest.csv')
df_ar_pl.head()

In [ ]:
# Brain maps — AntiRight ISC, ProLeft ISC, contrast, contrast (FDR-masked)
isc_scale = [float(min(mu_ar.min(), mu_pl.min())),
             float(max(mu_ar.max(), mu_pl.max()))]
cmax_ar_pl = float(np.abs(mu_ar - mu_pl).max())

print('AntiRight ISPC:')
brain_map(mu_ar, 'brain_isc_antiright', cmap='Reds', vminmax=isc_scale)

print('\nProLeft ISPC:')
brain_map(mu_pl, 'brain_isc_proleft', cmap='Reds', vminmax=isc_scale)

print('\nAntiRight − ProLeft (all parcels):')
brain_map(mu_ar - mu_pl, 'brain_antiright_vs_proleft',
          cmap='coolwarm', vminmax=(-cmax_ar_pl, cmax_ar_pl))

print('\nAntiRight − ProLeft (FDR significant only):')
diff_masked = np.where(rejected_ar_pl, mu_ar - mu_pl, np.nan)
brain_map(diff_masked, 'brain_antiright_vs_proleft_sig',
          cmap='coolwarm', vminmax=(-cmax_ar_pl, cmax_ar_pl))

## 12. Anti vs Pro Content Contrast

Collapses political affiliation and compares content by **valence**:
- **Anti** = AntiRight + AntiLeft (all outgroup-derogating / attack content)
- **Pro** = ProLeft + ProRight (all ingroup-affirming / supportive content)

Tests whether attack content (regardless of target party) elicits greater
neural alignment than supportive content.

**Method**: Same as Section 11 — paired t-test per parcel, FDR correction.

In [ ]:
# Merge into Anti and Pro groups (reuse merge_conditions)
from yy_fmri_kit.event_isc.contrast import merge_conditions

merged_valence = merge_conditions(
    patterns,
    {
        'anti': ['AntiLeft', 'AntiRight'],
        'pro':  ['ProLeft',  'ProRight'],
    }
)

t_anti_pro, p_anti_pro, mu_anti, mu_pro = ttest_contrast(
    merged_valence['anti'], merged_valence['pro']
)
rejected_anti_pro, p_fdr_anti_pro = fdr_correct(p_anti_pro, q=cfg.fdr_q)

print(f'Significant parcels (FDR q={cfg.fdr_q}): {rejected_anti_pro.sum()} / {len(rejected_anti_pro)}')
if rejected_anti_pro.any():
    for i in rejected_anti_pro.nonzero()[0]:
        print(f'  {parcel_names[i]:50s}  t={t_anti_pro[i]:.3f}  p_fdr={p_fdr_anti_pro[i]:.4f}')

In [ ]:
# Save results
df_ap = results_to_dataframe(parcel_names, t_anti_pro, p_anti_pro, rejected_anti_pro, p_fdr_anti_pro)
df_ap = df_ap.rename(columns={'r': 't_stat'})
df_ap['isc_anti'] = mu_anti
df_ap['isc_pro']  = mu_pro
df_ap['isc_diff'] = mu_anti - mu_pro
df_ap.to_csv(OUTPUT_DIR / 'anti_vs_pro_ttest.csv', index=False)
print('Saved anti_vs_pro_ttest.csv')
df_ap.head()

In [ ]:
# Brain maps — Anti ISC, Pro ISC, contrast, contrast (FDR-masked)
isc_scale_ap = [float(min(mu_anti.min(), mu_pro.min())),
                float(max(mu_anti.max(), mu_pro.max()))]
cmax_ap = float(np.abs(mu_anti - mu_pro).max())

print('Anti ISPC (AntiLeft + AntiRight):')
brain_map(mu_anti, 'brain_isc_anti', cmap='Reds', vminmax=isc_scale_ap)

print('\nPro ISPC (ProLeft + ProRight):')
brain_map(mu_pro, 'brain_isc_pro', cmap='Reds', vminmax=isc_scale_ap)

print('\nAnti − Pro (all parcels):')
brain_map(mu_anti - mu_pro, 'brain_anti_vs_pro',
          cmap='coolwarm', vminmax=(-cmax_ap, cmax_ap))

print('\nAnti − Pro (FDR significant only):')
diff_masked_ap = np.where(rejected_anti_pro, mu_anti - mu_pro, np.nan)
brain_map(diff_masked_ap, 'brain_anti_vs_pro_sig',
          cmap='coolwarm', vminmax=(-cmax_ap, cmax_ap))